# Bài 2: baseline R của RQ2 (ptLasso, Trans-GLM; QĐ68)

Notebook chạy bằng kernel **Python**: clone đúng commit đã push, ghi đầu vào bằng `scripts/export_rq2_inputs.py`, tự cài một bản R riêng
bằng micromamba (conda-forge) rồi gọi `scripts/r/rq2_r_baselines.R`; với `STAGE = 'all'` còn chạy `scripts/run_rq2_r_eval.py` để lấy panel
và đánh giá như MVP. Script R dừng nếu ptLasso khác 1.0.2 hoặc glmtrans khác 2.1.0.

| STAGE | Việc | Thời gian | Kết quả cần xem |
|---|---|---|---|
| `'stage1'` | Chỉ ptLasso giai đoạn 1 trên TCGA, hai không gian gen (GPL570, GPL6884) | vài phút + cài R | `stage1_summary.csv`: cột `size_S1`, báo GVHD ngay (Review 08/11 Mục 7 câu 3) |
| `'all'` | Giai đoạn 1 + giai đoạn 2 (α ∈ {0,75; 0,5; 0,25}) + Trans-GLM cho mọi lần chạy của 4 cohort, rồi đánh giá | khoảng 1–3 giờ | `RQ2_r/<cohort>/rq2_r_per_run.csv`, `rq2_r_diagnostics.csv` |

Mỗi phiên:
1. Language: **Python**. Accelerator: None. Bật **Internet** (clone, tải R, tải gói từ CRAN).
2. Add Input: **experiment-data-v3** (bản mới nhất). Không cần output của việc khác.
3. Ô cấu hình: `STAGE`, `REPO_COMMIT` (commit đã push), `INPUT_V3_DATASET`.
4. Save Version → Save & Run All. Kết quả: `/kaggle/working/rq2_r_<STAGE>_<commit7>_results.zip`.


In [ ]:
from pathlib import Path
import os, shutil, subprocess, sys, tarfile, time, urllib.request

STAGE = 'stage1'                     # 'stage1' (chỉ |S1|, chạy trước) hoặc 'all'
REPO_URL = 'https://github.com/datachain-uit/CN-23520899-LuanNT.git'
REPO_BRANCH = 'rq2-mvp'
REPO_COMMIT = None                   # commit đã push chứa scripts/r/rq2_r_baselines.R
INPUT_V3_DATASET = Path('/kaggle/input/experiment-data-v3')   # sửa theo đường dẫn trong panel Input
DATASETS = ['gse10245', 'gse30219', 'gse37745', 'gse42127']

REPO_DIR = Path('/tmp/CN-23520899-LuanNT')
WORK_ROOT = Path('/kaggle/working/kltn_data')
OUT_DIR = Path('/kaggle/working/rq2_r')
LIB_DIR = Path('/tmp/r_baselines_lib')
MAMBA_DIR = Path('/tmp/micromamba')
R_ENV = Path('/tmp/r_env')
MICROMAMBA_URL = 'https://micro.mamba.pm/api/micromamba/linux-64/latest'
CONDA_PACKAGES = ['r-base==4.5.3', 'r-glmnet==5.0', 'r-ggplot2', 'r-gridextra', 'r-matrix', 'r-caret', 'r-foreach', 'r-doparallel', 'r-assertthat', 'r-formatr']
assert STAGE in ('stage1', 'all'), STAGE
T0 = time.time()
print(sys.version)

In [ ]:
# Gate: clone đúng commit đã push; dữ liệu đúng layout; biến môi trường cho các script Python.
assert REPO_COMMIT, 'Điền REPO_COMMIT ở ô cấu hình'
if REPO_DIR.exists(): shutil.rmtree(REPO_DIR)
subprocess.run(['git', 'clone', '--quiet', '--branch', REPO_BRANCH, REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(['git', 'checkout', '--quiet', REPO_COMMIT], cwd=REPO_DIR, check=True)
commit_hash = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True).strip()
assert subprocess.check_output(['git', 'branch', '-r', '--contains', 'HEAD'], cwd=REPO_DIR, text=True).strip(), f'{commit_hash} chưa có trên remote'
for f in ('scripts/r/rq2_r_baselines.R', 'scripts/export_rq2_inputs.py', 'scripts/run_rq2_r_eval.py'):
    assert (REPO_DIR / f).is_file(), f'commit {commit_hash} không có {f}'
assert (INPUT_V3_DATASET / 'experiment-data-geo-target').is_dir(), f'Không thấy Kaggle Input: {INPUT_V3_DATASET}'
WORK_ROOT.mkdir(parents=True, exist_ok=True)
for folder in ('experiment-data-perm', 'experiment-data-geo-target', 'split_manifests'):
    link = WORK_ROOT / folder
    if not link.exists(): link.symlink_to(INPUT_V3_DATASET / folder, target_is_directory=True)
if OUT_DIR.exists(): shutil.rmtree(OUT_DIR)
OUT_DIR.mkdir(parents=True)
os.environ.update({'KLTN_DATA_ROOT': str(WORK_ROOT), 'KLTN_STRICT_N': '1', 'PYTHONUTF8': '1'})
for mod in ('numpy', 'pandas', 'scipy', 'sklearn', 'tables'):
    print(f'{mod:10s}', getattr(__import__(mod), '__version__', 'ok'))
(OUT_DIR / 'pip_freeze.txt').write_text(subprocess.run([sys.executable, '-m', 'pip', 'freeze'], capture_output=True, text=True).stdout, encoding='utf-8')
print('COMMIT', commit_hash, '| STAGE', STAGE)

In [ ]:
# Đầu vào cho R (Python): 5.000 gen lọc trên nguồn, nguồn đã z-score, đích log2 gốc, lần chạy, fold trong L_T, seed R.
INPUTS = WORK_ROOT / 'results' / 'survey_v2' / 'RQ2_r_inputs'
with open(OUT_DIR / 'export.log', 'w', encoding='utf-8') as log:
    export_status = subprocess.run([sys.executable, 'scripts/export_rq2_inputs.py', '--datasets', *DATASETS], cwd=REPO_DIR, stdout=log, stderr=subprocess.STDOUT).returncode
print('EXPORT EXIT', export_status)
print('\n'.join((OUT_DIR / 'export.log').read_text(encoding='utf-8', errors='replace').splitlines()[-12:]))

In [ ]:
# Cài R riêng bằng micromamba; lỗi được ghi lại, không dừng, để ô cuối vẫn lưu zip.
setup_error = None
micromamba = MAMBA_DIR / 'bin' / 'micromamba'
t0 = time.time()
if export_status == 0:
    try:
        MAMBA_DIR.mkdir(parents=True, exist_ok=True)
        archive_mm = MAMBA_DIR / 'micromamba.tar.bz2'
        urllib.request.urlretrieve(MICROMAMBA_URL, archive_mm)
        with tarfile.open(archive_mm) as tar:
            tar.extract('bin/micromamba', MAMBA_DIR)
        os.chmod(micromamba, 0o755)
        with open(OUT_DIR / 'setup_r.log', 'w', encoding='utf-8') as log:
            subprocess.run([str(micromamba), 'create', '--yes', '--quiet', '--prefix', str(R_ENV), '--override-channels', '--channel', 'conda-forge', *CONDA_PACKAGES],
                           stdout=log, stderr=subprocess.STDOUT, check=True, env={**os.environ, 'MAMBA_ROOT_PREFIX': str(MAMBA_DIR)})
        print(subprocess.check_output([str(R_ENV / 'bin' / 'Rscript'), '--version'], text=True, stderr=subprocess.STDOUT).strip())
    except Exception as exc:
        setup_error = repr(exc)
        (OUT_DIR / 'setup_error.txt').write_text(setup_error, encoding='utf-8')
        print('LỖI CÀI R:', setup_error)
print(f'Cài R: {(time.time() - t0) / 60:.1f} phút')

In [ ]:
# Script R: giai đoạn 1 (và với STAGE = 'all': giai đoạn 2, Trans-GLM). Không raise ở đây để kết quả luôn được lưu.
r_status = None
R_OUT = OUT_DIR / 'r_out'
if export_status == 0 and setup_error is None:
    env = {k: v for k, v in os.environ.items() if k not in ('LD_LIBRARY_PATH', 'R_HOME', 'PYTHONPATH')}
    env.update({'PATH': f"{R_ENV / 'bin'}:{env['PATH']}", 'R_BASELINE_LIB': str(LIB_DIR), 'REPO_COMMIT_HASH': commit_hash})
    t0 = time.time()
    with open(OUT_DIR / 'r_stdout.log', 'w', encoding='utf-8') as out, open(OUT_DIR / 'r_stderr.log', 'w', encoding='utf-8') as err:
        r_status = subprocess.run([str(R_ENV / 'bin' / 'Rscript'), 'scripts/r/rq2_r_baselines.R', str(INPUTS), str(R_OUT), STAGE, *DATASETS],
                                  stdout=out, stderr=err, env=env, cwd=REPO_DIR).returncode
    print('R EXIT', r_status, f'sau {(time.time() - t0) / 60:.1f} phút')
    print('\n'.join((OUT_DIR / 'r_stdout.log').read_text(encoding='utf-8', errors='replace').splitlines()[-30:]))
    if (R_OUT / 'stage1_summary.csv').exists():
        print((R_OUT / 'stage1_summary.csv').read_text(encoding='utf-8'))

In [ ]:
# STAGE = 'all': panel (α, β_eff, F-test bổ sung, cờ suy biến) và đánh giá LR-L2 như MVP.
eval_status = None
if STAGE == 'all' and r_status == 0:
    with open(OUT_DIR / 'eval.log', 'w', encoding='utf-8') as log:
        eval_status = subprocess.run([sys.executable, 'scripts/run_rq2_r_eval.py', '--inputs', str(INPUTS), '--r-out', str(R_OUT), '--datasets', *DATASETS,
                                      '--save-scores'], cwd=REPO_DIR, stdout=log, stderr=subprocess.STDOUT).returncode
    print('EVAL EXIT', eval_status)
    print('\n'.join((OUT_DIR / 'eval.log').read_text(encoding='utf-8', errors='replace').splitlines()[-12:]))

In [ ]:
# Lưu kết quả rồi mới báo lỗi. Không đưa ma trận biểu hiện (*.csv.gz) vào zip; chúng tái tạo được từ commit + dữ liệu.
import json
stage_dir = OUT_DIR / 'results_survey_v2'
for sub in ('RQ2_r_inputs', 'RQ2_r'):
    src = WORK_ROOT / 'results' / 'survey_v2' / sub
    if src.exists():
        shutil.copytree(src, stage_dir / sub, ignore=shutil.ignore_patterns('*.csv.gz'), dirs_exist_ok=True)
info = {'stage': STAGE, 'commit_hash': commit_hash, 'export_status': export_status, 'setup_error': setup_error, 'r_status': r_status,
        'eval_status': eval_status, 'hours': round((time.time() - T0) / 3600, 2), 'finished_at': time.strftime('%Y-%m-%d %H:%M:%S %Z')}
(OUT_DIR / 'run_info.json').write_text(json.dumps(info, indent=2), encoding='utf-8')
archive = shutil.make_archive(f'/kaggle/working/rq2_r_{STAGE}_{commit_hash[:7]}_results', 'zip', root_dir=OUT_DIR.parent, base_dir=OUT_DIR.name)
print('ARCHIVE', archive, os.path.getsize(archive), 'byte', info)
if export_status != 0: raise RuntimeError('export_rq2_inputs.py lỗi: xem export.log trong zip')
if setup_error is not None: raise RuntimeError(f'Không cài được R: {setup_error}')
if r_status != 0: raise RuntimeError(f'Script R thoát mã {r_status} (2 = sai phiên bản gói): xem r_stdout.log, r_stderr.log')
if STAGE == 'all' and eval_status != 0: raise RuntimeError('run_rq2_r_eval.py lỗi: xem eval.log')
print(f'RQ2_R_{STAGE.upper()}_OK')